# TATN 预训练阶段完整复现
**目标**：对 5 个温度（25°C、10°C、0°C、-10°C、-20°C）分别预训练，保存模型和结果，对照论文 Table II / III。

**运行前**：Runtime → Change runtime type → GPU

## Step 1：挂载 Google Drive，设置路径

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# ===== 根据你的 Drive 结构修改这里 =====
DRIVE_RAW_DATA = '/content/drive/MyDrive/Research/mining review/TATN/Panasonic 18650PF Data'
DRIVE_RESULTS  = '/content/drive/MyDrive/Research/mining review/TATN/results/pretrain'
WORK_DIR = '/content/TATN'
# =========================================

print('根目录内容：')
for item in sorted(os.listdir(DRIVE_RAW_DATA)):
    print(' ', item)

## Step 2：拉取代码

In [ ]:
import shutil
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
!git clone https://github.com/zhangcastle/TATN.git /content/TATN -q
os.chdir(WORK_DIR)
print('工作目录:', os.getcwd())

## Step 3：安装依赖

In [ ]:
!pip install scipy tqdm scikit-learn -q
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 4：探索原始数据结构

In [ ]:
def find_drive_cycles_dir(base, temp_folder):
    """在 base/temp_folder 下找 Drive cycles 子目录（大小写不敏感）"""
    candidates = [
        os.path.join(base, temp_folder, 'Drive cycles'),
        os.path.join(base, temp_folder, 'Drive Cycles'),
        os.path.join(base, temp_folder, 'drive cycles'),
        os.path.join(base, temp_folder),   # 文件直接在温度目录下
    ]
    for c in candidates:
        if os.path.exists(c) and any(f.endswith('.mat') for f in os.listdir(c)):
            return c
    return None

# 25degC 特殊：在嵌套的子目录里
temp_raw = {
    '25':  find_drive_cycles_dir(os.path.join(DRIVE_RAW_DATA, 'Panasonic 18650PF Data'), '25degC'),
    '10':  find_drive_cycles_dir(DRIVE_RAW_DATA, '10degC'),
    '0':   find_drive_cycles_dir(DRIVE_RAW_DATA, '0degC'),
    'n10': find_drive_cycles_dir(DRIVE_RAW_DATA, '-10degC'),
    'n20': find_drive_cycles_dir(DRIVE_RAW_DATA, '-20degC'),
}

for temp, path in temp_raw.items():
    print(f'\n=== {temp} ===')
    if path:
        print(f'  路径: {path}')
        for f in sorted(os.listdir(path)):
            if f.endswith('.mat'):
                print(' ', f)
    else:
        print('  [未找到，请手动设置路径]')

## Step 5：数据处理

注意：
- HWFTa + HWFTb 会自动合并为 HWFET
- 含多个 drive cycle 的合并文件（如 LA92_NN）会自动跳过

In [ ]:
import scipy.io as sio
import numpy as np

# 关键词映射，HWFTa/HWFTb 合并为 HWFET
CYCLE_KW = {
    'Cycle_1': ['Cycle_1', 'Cycle1'],
    'Cycle_2': ['Cycle_2', 'Cycle2'],
    'Cycle_3': ['Cycle_3', 'Cycle3'],
    'Cycle_4': ['Cycle_4', 'Cycle4'],
    'NN':      ['_NN_', '_NN.'],
    'US06':    ['US06'],
    'HWFET':   ['HWFET', 'HWFTa', 'HWFTb'],
    'LA92':    ['LA92'],
    'UDDS':    ['UDDS'],
}

def match_cycle(fname):
    matched = []
    for cycle, kws in CYCLE_KW.items():
        if any(kw in fname for kw in kws):
            matched.append(cycle)
    if len(matched) == 1:
        return matched[0]
    if len(matched) > 1:
        return f'SKIP({','.join(matched)})'
    return None

def read_mat_file(fpath):
    data = sio.loadmat(fpath)
    if 'meas' in data:
        m = data['meas']
        return (m['Time'][0][0].flatten(), m['Current'][0][0].flatten(),
                m['Voltage'][0][0].flatten(), m['Battery_Temp_degC'][0][0].flatten(),
                m['Ah'][0][0].flatten())
    return (data['time'].flatten(), data['current'].flatten(),
            data['voltage'].flatten(), data['temp'].flatten(), data['ah'].flatten())

def process_temp(raw_path, temp_label, out_base, interval=10, train_ratio=0.3):
    out_path = os.path.join(out_base, temp_label)
    os.makedirs(out_path, exist_ok=True)
    mat_files = sorted([f for f in os.listdir(raw_path) if f.endswith('.mat')])
    all_data = {}   # cycle_name -> (cur, vol, tmp, ah)
    for fname in mat_files:
        cycle = match_cycle(fname)
        if cycle is None:
            print(f'  [未识别] {fname}')
            continue
        if cycle.startswith('SKIP'):
            print(f'  [跳过合并文件] {fname}  ({cycle})')
            continue
        t, cur, vol, tmp, ah = read_mat_file(os.path.join(raw_path, fname))
        # 降采样
        t2 = t[::interval]; cur=cur[::interval]; vol=vol[::interval]
        tmp=tmp[::interval]; ah=ah[::interval]
        # 找连续段起点
        idx = next((i for i in range(len(t2)-1) if t2[i+1]-t2[i] < 2), 0)
        seg = (cur[idx:], vol[idx:], tmp[idx:], ah[idx:])
        # 合并同 cycle 的数据（如 HWFTa + HWFTb）
        if cycle in all_data:
            all_data[cycle] = tuple(np.concatenate([all_data[cycle][i], seg[i]]) for i in range(4))
            print(f'  [{temp_label}] 合并: {fname} -> {cycle}  total={len(all_data[cycle][0])}')
        else:
            all_data[cycle] = seg
            print(f'  [{temp_label}] 读取: {fname} -> {cycle}  n={len(seg[0])}')
    if not all_data:
        print(f'[{temp_label}] 无数据，跳过'); return
    # 全局 min/max（按温度，论文方法）
    def grange(i):
        v = np.concatenate([d[i] for d in all_data.values()])
        return v.min(), v.max()
    ranges = [grange(i) for i in range(4)]
    names = ['current','voltage','temp','ah']
    print(f'\n  [{temp_label}] 归一化范围:')
    for n, r in zip(names, ranges):
        print(f'    {n}: [{r[0]:.4f}, {r[1]:.4f}]')
    norm = lambda x, r: (x - r[0]) / (r[1] - r[0])
    for cycle, (cur, vol, tmp, ah) in all_data.items():
        cn,vn,tn,an = norm(cur,ranges[0]),norm(vol,ranges[1]),norm(tmp,ranges[2]),norm(ah,ranges[3])
        sp = int(len(an)*train_ratio)
        for suf, sl in [('train',slice(None,sp)),('test',slice(sp,None))]:
            sio.savemat(os.path.join(out_path, f'{temp_label}{cycle}_{suf}.mat'),
                {'current':cn[sl].reshape(-1,1),'voltage':vn[sl].reshape(-1,1),
                 'temp':tn[sl].reshape(-1,1),'ah':an[sl].reshape(-1,1)})
        print(f'  saved {cycle}  train={sp} test={len(an)-sp}')
    print(f'[{temp_label}] done -> {out_path}\n')

OUTPUT_BASE = os.path.join(WORK_DIR, 'normalized_data', 'Pan')
for temp_label, raw_path in temp_raw.items():
    print(f'========== {temp_label}°C ==========')
    if raw_path:
        process_temp(raw_path, temp_label, OUTPUT_BASE)
    else:
        print('路径未找到，跳过\n')
print('全部处理完成')

## Step 6：预训练（5 个温度，每个温度跑完立刻保存到 Drive）

In [ ]:
import sys, time, shutil, glob
import torch, torch.nn as nn, torch.optim as optim
sys.path.insert(0, WORK_DIR)
import importlib
import mydata, models
from pretrain import pretrain

mydata.Pan_data_path = OUTPUT_BASE + '/'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
EPOCHS, BATCH_SIZE, LR, EVAL_INTERVAL = 2000, 64, 0.001, 200
TEMPS = ['25', '10', '0', 'n10', 'n20']
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Device: {DEVICE}  Epochs: {EPOCHS}')

In [ ]:
importlib.reload(mydata)
importlib.reload(models)
summary = {}

for temp in TEMPS:
    print(f'\n{"="*55}\n预训练: {temp}°C\n{"="*55}')
    if not os.path.exists(os.path.join(OUTPUT_BASE, temp)):
        print('数据不存在，跳过'); continue
    importlib.reload(models)
    mdls = {
        'conv':models.conv(),'lstm':models.lstm(),'fc':models.fc(),
        'regression':models.regression(),'conv_s':models.conv(),
        'lstm_s':models.lstm(),'fc_s':models.fc(),'regression_s':models.regression(),
        'discriminator':models.Discriminator(),
    }
    opts = {k: optim.Adam(mdls[k].parameters(), lr=LR)
            for k in ['conv','lstm','fc','regression','discriminator']}
    t0 = time.time()
    pretrain(
        rundir=f'pretrain_{temp}', source_temp=temp, target_temp=temp,
        source_data_path=OUTPUT_BASE+'/', source_train_set=mydata.Pan_train_set,
        source_test_set=mydata.Pan_test_set, models=mdls,
        criterion=nn.MSELoss(reduction='sum'), optimizers=opts,
        batch_size=BATCH_SIZE, epochs=EPOCHS, eval_interval=EVAL_INTERVAL,
        seed=100, device_type=DEVICE, ifsave=True, load_model=False,
    )
    elapsed = (time.time()-t0)/60
    # 每个温度跑完立刻保存到 Drive
    pts = glob.glob(os.path.join(WORK_DIR, f'run/pretrain_{temp}*/saved_model/best.pt'))
    if pts:
        dest = os.path.join(DRIVE_RESULTS, f'pre-{temp}.pt')
        shutil.copy(pts[0], dest)
        print(f'[{temp}°C] 模型已保存到 Drive: {dest}')
    else:
        print(f'[{temp}°C] 未找到 best.pt！')
    summary[temp] = elapsed
    print(f'[{temp}°C] 完成，耗时 {elapsed:.1f} min')

print('\n========== 全部完成 ==========')
for t, m in summary.items():
    print(f'  {t}°C: {m:.1f} min')
print('\n论文参考值: 训练 MAE=0.294% RMSE=0.366% | 测试平均 MAE=1.09% RMSE=1.44%')